In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab.models.galaxy_zoo import (
    GalaxyZooData,
    GalaxyZooImages,
    GalaxyZooDataset,
)
from ugdatalab.models.galaxy_zoo.constants import N_LABELS, LABEL_COLUMNS, LABEL_DESCRIPTIVE
from ugdatalab.models.galaxy_zoo.images_pipeline import _load_image
from ugdatalab.methods.neural_network.cnn import baseline_rmse

import plotters

# Galaxy Image Classification — Preprocessing

This notebook handles image preprocessing (Tasks 10–13):
1. **Task 10** — Crop and resize images to reduce memory by ~30×
2. **Task 11** — Set up efficient batch loading via PyTorch DataLoader
3. **Task 12** — Split into 80% training / 20% validation and verify label distributions
4. **Task 13** — Establish baseline model (mean prediction) RMSE

In [ ]:
CSV_PATH = Path("data/training_classifications.csv")
IMAGE_DIR = Path("data/training_images")

gz = GalaxyZooData(CSV_PATH)
labels_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = labels_data["labels"]
galaxy_ids = labels_data["galaxy_ids"]

print(f"Galaxies: {len(gz)}")
print(f"Labels shape: {labels.shape}")

## Task 10 — Image Downsizing

The raw SDSS images are ~424×424 pixels, most of which is empty sky. We reduce memory and computation by:

1. **Center-cropping**: removing 25% of the border on each side (keeping the central 50%, which contains the galaxy). This is justified because SDSS cutouts are centered on the target and the outer border is almost always empty sky.
2. **Resampling**: resizing the cropped image to a smaller pixel grid.

The model input size is **96×96**. We cache images at the slightly larger **136×136** ($96 \times \sqrt{2}$, rounded up): this is the smallest size for which an arbitrary rotation of the cached image still fully covers a centered 96×96 inscribed square. Training and evaluation pipelines apply a downstream `CenterCrop(96)` (composed after any rotation augmentation), so the model always sees corner-artifact-free 96×96 input. We use 96 for the model rather than the nominal 64 so that ResNet-18 keeps its standard 7×7 stem; 96 also divides cleanly through four pooling stages (96 → 48 → 24 → 12 → 6).

In [ ]:
CROP_FRACTION = 0.25
# Rotation-safe cache size: ceil(96 * sqrt(2)). Downstream transforms
# rotate then CenterCrop(96), guaranteeing no black-corner artifacts.
CACHE_SIZE = 136

# Show before/after for a few example images
n_compare = 4
rng = np.random.default_rng(42)
compare_idx = rng.choice(len(gz), size=n_compare, replace=False)
compare_ids = galaxy_ids[compare_idx]

images_before = [_load_image(IMAGE_DIR / f"{gid}.jpg") for gid in compare_ids]

from ugdatalab.models.galaxy_zoo.images_pipeline import _crop_center, _resize
images_after = [_resize(_crop_center(img, CROP_FRACTION), CACHE_SIZE) for img in images_before]

axes = plotters.plot_image_comparison(images_before, images_after, compare_ids)
plt.show()

# Report size reduction
h_orig = images_before[0].shape[0]
reduction = (h_orig ** 2) / (CACHE_SIZE ** 2)
print(f"Original: {h_orig}x{h_orig} = {h_orig**2:,} pixels")
print(f"After crop+resize: {CACHE_SIZE}x{CACHE_SIZE} = {CACHE_SIZE**2:,} pixels")
print(f"Reduction factor: {reduction:.1f}x")

### Preprocess and save all images

We now crop and resize all images and save the result as a compressed numpy archive. This takes a few minutes but only needs to be done once — subsequent notebooks load the preprocessed arrays directly.

In [ ]:
gz_images = GalaxyZooImages(
    source=gz,
    image_dir=IMAGE_DIR,
    crop_fraction=CROP_FRACTION,
    target_size=CACHE_SIZE,
)

print(f"Preprocessed images shape: {gz_images.images.shape}")
print(f"Memory: {gz_images.images.nbytes / 1e9:.2f} GB")

np.savez_compressed(
    "artifacts/galaxy_zoo_images.npz",
    images=gz_images.images,
    galaxy_ids=galaxy_ids,
)
print("Saved artifacts/galaxy_zoo_images.npz")

## Task 12 — Train/Validation Split

We split the data 80/20 into training and validation sets using a random permutation with a fixed seed. After splitting, we compare the normalized label distributions of the two sets to ensure there are no systematic differences — the split should produce statistically indistinguishable distributions for all 37 labels.

In [ ]:
rng = np.random.default_rng(42)
idx = rng.permutation(len(gz))
n_train = int(0.8 * len(gz))
train_idx = np.sort(idx[:n_train])
val_idx = np.sort(idx[n_train:])
train_labels = gz.labels[train_idx]
val_labels = gz.labels[val_idx]

print(f"Training set: {len(train_idx)} galaxies ({len(train_idx)/len(gz)*100:.0f}%)")
print(f"Validation set: {len(val_idx)} galaxies ({len(val_idx)/len(gz)*100:.0f}%)")

# Compare distributions
axes = plotters.plot_split_distributions(
    train_labels, val_labels, LABEL_COLUMNS, LABEL_DESCRIPTIVE,
)
plt.show()

## Task 13 — Baseline Model

Before training any neural network, we establish a baseline: predict the training-set mean label for every image. This is the simplest possible model — it ignores the image entirely and always predicts the same label vector. Any useful CNN must outperform this baseline.

The loss function throughout this lab is the **root mean squared error** (RMSE), defined as:

$$L_{\mathrm{RMSE}} = \sqrt{\frac{1}{N_{\mathrm{galaxies}} \cdot N_{\mathrm{labels}}} \sum_i \sum_j (\ell_{\mathrm{true},ij} - \ell_{\mathrm{pred},ij})^2}$$

In [ ]:
train_rmse, val_rmse = baseline_rmse(train_labels, val_labels)
print(f"Baseline RMSE (mean prediction):")
print(f"  Training:   {train_rmse:.4f}")
print(f"  Validation: {val_rmse:.4f}")

### Save split indices and preprocessed data

In [ ]:
np.savez_compressed(
    "artifacts/split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    baseline_train_rmse=train_rmse,
    baseline_val_rmse=val_rmse,
)
print("Saved artifacts/split_indices.npz")
print(f"  train_idx: {train_idx.shape}")
print(f"  val_idx: {val_idx.shape}")